# Week 81, Day 3: Queues

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w081d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from collections import deque

queue = deque()
for n in range(1, 6):
    queue.append(f"email {n}")
print(queue.popleft(), queue.popleft())
queue.append("email 6")
print(list(queue), len(queue))

In [ ]:
messages = {1: {"body": "welcome email to 7", "receives": 0}}
invisible_until = {}

def receive(now, timeout=30):
    for mid, m in messages.items():
        if invisible_until.get(mid, 0) <= now:
            m["receives"] += 1
            invisible_until[mid] = now + timeout
            return mid, m["body"]
    return None

print(receive(0))           # a worker takes it...
print(receive(10))          # ...so nobody else can while it's working
print(receive(31))          # the worker never acked: it reappears
del messages[1]             # this time the worker finishes and acks
print(receive(70))

In [ ]:
processed = set()
balance = 0

def handle(message):
    global balance
    if message["id"] in processed:
        return "duplicate"
    balance += message["amount"]
    processed.add(message["id"])
    return "applied"

for m in [{"id": "pay-1", "amount": 50}, {"id": "pay-2", "amount": 20}, {"id": "pay-1", "amount": 50}]:
    print(m["id"], handle(m))
print("balance:", balance)

> **Key idea.** Queues let producers hand work to consumers: requests stay fast, failures downstream don't fail requests, bursts are smoothed, and failed work is retried. Queue work the response doesn't depend on. Bound queues and push back (refuse, 429, 503) instead of growing without limit. At-most-once deletes on delivery and can lose messages; at-least-once hides a message during a visibility timeout and deletes it on ack, so nothing is lost but duplicates happen; exactly-once is achieved in effect with idempotent consumers. Retry with backoff and move poison messages to a dead-letter queue after a maximum number of receives. Make consumers idempotent: naturally, or by recording processed message ids. Pub/sub gives each subscription its own copy; partitioned logs keep per-key order by hashing keys stably (zlib.crc32, not hash()) and tracking offsets per partition.

### Exercise: A queue that pushes back

Write a class `BoundedQueue(capacity)` with `put(item)` (add to the back and return `True`, or return `False` and count a rejection if the queue is full), `get()` (remove and return the front item, or `None` if empty), `__len__`, and `stats()` returning `{"accepted": A, "rejected": R, "max_length": M}`, where M is the longest the queue has been. Use `collections.deque`. Then `submit(queue, items)`: put each item and return the list of HTTP statuses a web endpoint would answer for each: `202` (accepted for later) or `503` (try again later).

In [ ]:
from collections import deque

class BoundedQueue:
    ___

def submit(queue, items):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_bounded():\n    \"full means no\"\n    q = BoundedQueue(3)\n    assert submit(q, [\"a\", \"b\", \"c\", \"d\", \"e\"]) == [202, 202, 202, 503, 503]\n    assert q.get() == \"a\" and len(q) == 2\n    assert q.put(\"f\") is True and q.put(\"g\") is False\n    assert [q.get() for _ in range(4)] == [\"b\", \"c\", \"f\", None]\n    assert q.stats() == {\"accepted\": 4, \"rejected\": 3, \"max_length\": 3}")

### Exercise: Acks, timeouts and dead letters

Write a class `VisibilityQueue(timeout, max_receives)`, like Amazon SQS. `send(body)` adds a message with the next id (from 1) and returns the id. `receive(now)` returns `(id, body)` for the oldest message that's visible (not being worked on, or whose visibility timeout has run out), making it invisible until `now + timeout` and counting a receive; but a message that has already been received `max_receives` times is moved to `self.dead_letters` (a list of `(id, body)`) instead of being handed out again. Return `None` if nothing is visible. `ack(message_id)` deletes a message and returns `True`, or returns `False` if there's no such message (already deleted or dead-lettered).

In [ ]:
class VisibilityQueue:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_visibility():\n    \"at-least-once delivery\"\n    q = VisibilityQueue(timeout=30, max_receives=2)\n    assert q.send(\"email 7\") == 1 and q.send(\"email 8\") == 2\n    assert q.receive(0) == (1, \"email 7\") and q.receive(0) == (2, \"email 8\") and q.receive(5) is None\n    assert q.ack(2) is True and q.ack(2) is False\n    assert q.receive(29) is None and q.receive(30) == (1, \"email 7\")\n    assert q.receive(70) is None and q.dead_letters == [(1, \"email 7\")]\n    assert q.ack(1) is False\n    q.send(\"email 9\")\n    assert q.receive(70) == (3, \"email 9\")")

### Exercise: An idempotent consumer

Write a class `Deduplicator(clock, keep_for)` with `seen(message_id)`: `True` if that id was recorded less than `keep_for` seconds ago; otherwise record it now and return `False`. Old records are forgotten (purge any older than `keep_for` on each call). Then `apply_payments(messages, dedup)`: a dict of account to balance, from messages `{"id", "account", "amount"}`, skipping any whose id `dedup` has seen, and a list of the ids skipped.

In [ ]:
class Deduplicator:
    ___

def apply_payments(messages, dedup):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_dedup():\n    \"duplicates do no harm\"\n    class Clock:\n        t = 0\n        def __call__(self):\n            return self.t\n    clock = Clock()\n    d = Deduplicator(clock, keep_for=3600)\n    msgs = [{\"id\": \"p1\", \"account\": \"ada\", \"amount\": 50}, {\"id\": \"p2\", \"account\": \"bob\", \"amount\": 20},\n            {\"id\": \"p1\", \"account\": \"ada\", \"amount\": 50}, {\"id\": \"p3\", \"account\": \"ada\", \"amount\": -10}]\n    assert apply_payments(msgs, d) == ({\"ada\": 40, \"bob\": 20}, [\"p1\"])\n    clock.t = 3599\n    assert d.seen(\"p2\") is True\n    clock.t = 3600\n    assert d.seen(\"p1\") is False and len(d.records) == 1")

### Exercise: Publish and subscribe

Write a class `Broker` with `subscribe(topic, name)` (create a subscription with its own queue; subscribing the same name twice changes nothing), `publish(topic, message)` (add a copy of the message to every subscription of that topic, returning how many got it), `pull(topic, name, max_messages=10)` (remove and return up to that many messages from that subscription's queue, oldest first) and `backlog()` (a dict from `"topic/name"` to queue length, sorted by key). Publishing to a topic with no subscribers delivers to nobody (return `0`).

In [ ]:
from collections import deque

class Broker:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_pubsub():\n    \"every subscription gets its own copy\"\n    b = Broker()\n    b.subscribe(\"ticket.resolved\", \"email\")\n    b.subscribe(\"ticket.resolved\", \"chat\")\n    b.subscribe(\"ticket.resolved\", \"chat\")\n    b.subscribe(\"ticket.created\", \"search\")\n    assert b.publish(\"ticket.resolved\", {\"id\": 7}) == 2 and b.publish(\"ticket.resolved\", {\"id\": 8}) == 2\n    assert b.publish(\"ticket.deleted\", {\"id\": 9}) == 0\n    assert b.pull(\"ticket.resolved\", \"email\") == [{\"id\": 7}, {\"id\": 8}]\n    assert b.pull(\"ticket.resolved\", \"chat\", 1) == [{\"id\": 7}]\n    assert b.backlog() == {\"ticket.created/search\": 0, \"ticket.resolved/chat\": 1, \"ticket.resolved/email\": 0}\n    m = {\"id\": 10}\n    b.publish(\"ticket.resolved\", m)\n    m[\"id\"] = 99\n    assert b.pull(\"ticket.resolved\", \"email\") == [{\"id\": 10}]")

### Exercise: A partitioned log

Write a class `PartitionedLog(partitions)`, like a Kafka topic. `partition_for(key)` is `zlib.crc32(key.encode()) % partitions`. `append(key, value)` adds `(key, value)` to the end of that key's partition and returns `(partition, offset)`, where the offset is its position in that partition (from 0). `read(group, partition, max_records=10)` returns up to that many records from the group's current offset in that partition and advances the group's offset; each consumer group has its own offsets, starting at 0. `lag(group)` returns a dict from partition number to how many records the group hasn't read yet. Records are never removed by reading.

In [ ]:
import zlib

class PartitionedLog:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_log():\n    \"per-key order, per-group offsets\"\n    import zlib\n    log = PartitionedLog(3)\n    assert log.partition_for(\"ticket:1\") == zlib.crc32(b\"ticket:1\") % 3 == 1\n    events = [(\"ticket:1\", \"created\"), (\"ticket:2\", \"created\"), (\"ticket:1\", \"assigned\"), (\"ticket:1\", \"resolved\"), (\"ticket:5\", \"created\")]\n    places = [log.append(k, v) for k, v in events]\n    assert places[:4] == [(1, 0), (2, 0), (1, 1), (1, 2)]\n    p = log.partition_for(\"ticket:1\")\n    assert log.read(\"email\", p, 2) == [(\"ticket:1\", \"created\"), (\"ticket:1\", \"assigned\")]\n    assert log.read(\"email\", p) == [(\"ticket:1\", \"resolved\")] + [r for r in [events[4]] if log.partition_for(r[0]) == p]\n    assert log.read(\"stats\", p, 1) == [(\"ticket:1\", \"created\")]\n    lag = log.lag(\"stats\")\n    assert lag[p] == len(log.partitions[p]) - 1 and sum(lag.values()) == 4\n    assert log.lag(\"new-group\") == {i: len(log.partitions[i]) for i in range(3)}")

### Exercise: How many workers?

Simulate a queue drained by workers, minute by minute. Write `drain(arrivals, workers, per_worker)`: `arrivals[i]` messages arrive in minute `i`; each minute, after the arrivals, the workers process up to `workers * per_worker` messages. Return `(peak_backlog, minutes_to_empty)`, where `peak_backlog` is the largest backlog at the end of any minute and `minutes_to_empty` is the number of minutes (counting from the first) until the backlog is empty at the end of a minute **and** no more arrivals are due, continuing past the end of `arrivals` as long as needed. Then `workers_for(arrivals, per_worker, max_minutes)`: the fewest workers (from 1) that empty it within `max_minutes`; since the queue can't be empty before the last arrivals are in, raise `ValueError("max_minutes is shorter than the arrivals")` if `max_minutes` is less than `len(arrivals)`.

In [ ]:
def drain(arrivals, workers, per_worker):
    ___

def workers_for(arrivals, per_worker, max_minutes):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_drain():\n    \"a burst of sign-ups\"\n    burst = [1000, 50, 50, 50, 50]\n    assert drain(burst, 1, 100) == (900, 12) and drain(burst, 2, 100) == (800, 6)\n    assert drain(burst, 5, 100) == (500, 5)\n    assert drain([10, 10], 1, 100) == (0, 2) and drain([], 3, 10) == (0, 0)\n    assert workers_for(burst, 100, 6) == 2 and workers_for(burst, 100, 30) == 1 and workers_for(burst, 100, 5) == 3\n    try:\n        workers_for(burst, 100, 4)\n    except ValueError as e:\n        assert str(e) == \"max_minutes is shorter than the arrivals\"\n    else:\n        raise AssertionError(\"no number of workers can finish before the arrivals do\")")

### Bug hunt: A message lost in a crash

This worker acknowledges each message **before** doing the work, so when it crashes on message 2, that message is already gone, and this prints `still queued: []`. Acknowledge only after the work succeeds, so the failed message stays for another worker: `still queued: ['email 2']`.

In [ ]:
queue = ["email 1", "email 2", "email 3"]
done = []

def work(message):
    if message == "email 2":
        raise ConnectionError("mail server down")
    done.append(message)

for message in list(queue):
    queue.remove(message)          # ack
    try:
        work(message)
    except ConnectionError:
        pass
print("still queued:", queue)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"ack after the work\"\n    expect_output(\"still queued: ['email 2']\")")

### Bug hunt: A poison message

A message that always fails is retried forever, so the worker never gets past it and spends every attempt on it: this prints `attempts on bad: 20, good processed: 0`. Give up after 3 attempts by moving it to a dead-letter list, so the good message gets processed: `attempts on bad: 3, good processed: 1, dead letters: ['bad']`.

In [ ]:
from collections import deque

queue = deque(["bad", "good"])
attempts = {"bad": 0, "good": 0}
processed, dead = [], []

for _ in range(20):
    if not queue:
        break
    message = queue.popleft()
    attempts[message] += 1
    if message == "bad":
        queue.appendleft(message)      # try again
        continue
    processed.append(message)
print(f"attempts on bad: {attempts['bad']}, good processed: {len(processed)}")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a dead-letter queue\"\n    expect_output(\"attempts on bad: 3, good processed: 1, dead letters: ['bad']\")")

### Bug hunt: A priority queue that crashed

Jobs are stored in a heap as `(priority, job)`, so two jobs with the same priority make Python compare the job **dicts**, which raises `TypeError`. Add a counter as a tie-breaker between the priority and the job (`(priority, count, job)`), so equal priorities keep their arrival order and this prints `['refund', 'email', 'report']`.

In [ ]:
import heapq

jobs = []
for priority, job in [(1, {"name": "refund"}), (2, {"name": "email"}), (2, {"name": "report"})]:
    heapq.heappush(jobs, (priority, job))
print([heapq.heappop(jobs)[-1]["name"] for _ in range(3)])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a tie-breaker\"\n    expect_output(\"['refund', 'email', 'report']\")")

### Puzzle: A careful worker

Arrange a worker loop that receives a message, skips duplicates, does the work, and acknowledges only after it succeeds. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # "at-most-once" or "at-least-once": can lose a message if a worker crashes
q = ___   # what a consumer must be, so a duplicate message does no harm
r = ___   # where messages go after failing too many times (three words)
s = ___   # True or False: Python's hash("ticket:1") is the same in every process
t = ___   # the HTTP status for "the queue is full, try again later"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"at-most-once\" and q == \"idempotent\" and r.lower() in (\"dead-letter queue\", \"dead letter queue\"), \"p, q and r\"\n    assert s is False and t == 503, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a background job runner

Write a class `JobRunner(handlers, max_attempts=4, base_delay=10)`: `handlers` maps a job type to a function that takes the payload and returns nothing or raises. `enqueue(job_type, payload, now, key=None)` adds a job due at `now` and returns its id (from 1), unless `key` is given and a job with that key was already enqueued (whatever its state): then it returns the existing job's id and adds nothing. An unknown job type raises `ValueError(f"no handler for {job_type}")`. `tick(now, batch=10)` runs up to `batch` jobs that are pending and due (`run_at <= now`), in order of `run_at` then id. A job that succeeds becomes `"done"`. A job that raises is retried: its attempt count rises and it's due again at `now + base_delay * 2 ** (attempts - 1)`; once it has failed `max_attempts` times it becomes `"dead"`. `tick` returns a list of log lines: `JOB_ID TYPE done`, `JOB_ID TYPE retry in Ns: MESSAGE` or `JOB_ID TYPE dead: MESSAGE`. `counts()` returns a dict of how many jobs are `pending`, `done` and `dead`.

In [ ]:
class JobRunner:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_runner():\n    \"retries, backoff and dead jobs\"\n    sent, failures = [], {\"n\": 0}\n    def email(payload):\n        if payload[\"to\"] == \"nobody@nowhere\":\n            raise ValueError(\"no such mailbox\")\n        sent.append(payload[\"to\"])\n    def webhook(payload):\n        failures[\"n\"] += 1\n        if failures[\"n\"] <= 2:\n            raise ConnectionError(\"receiver down\")\n    r = JobRunner({\"email\": email, \"webhook\": webhook}, max_attempts=3)\n    assert r.enqueue(\"email\", {\"to\": \"ada@example.org\"}, 0, key=\"welcome-1\") == 1\n    assert r.enqueue(\"email\", {\"to\": \"ada@example.org\"}, 0, key=\"welcome-1\") == 1\n    assert r.enqueue(\"webhook\", {\"ticket\": 7}, 0) == 2\n    assert r.enqueue(\"email\", {\"to\": \"nobody@nowhere\"}, 0) == 3\n    assert r.tick(0) == [\"1 email done\", \"2 webhook retry in 10s: receiver down\", \"3 email retry in 10s: no such mailbox\"]\n    assert r.tick(5) == []\n    assert r.tick(10) == [\"2 webhook retry in 20s: receiver down\", \"3 email retry in 20s: no such mailbox\"]\n    assert r.tick(30) == [\"2 webhook done\", \"3 email dead: no such mailbox\"]\n    assert r.counts() == {\"pending\": 0, \"done\": 2, \"dead\": 1} and sent == [\"ada@example.org\"]\n    try:\n        r.enqueue(\"sms\", {}, 0)\n    except ValueError as e:\n        assert str(e) == \"no handler for sms\"\n\ndef test_order_and_batch():\n    \"due jobs by time, then id, a batch at a time\"\n    done = []\n    r = JobRunner({\"x\": lambda p: done.append(p)})\n    r.enqueue(\"x\", \"late\", 50)\n    for n in range(5):\n        r.enqueue(\"x\", n, 0)\n    assert r.tick(10, batch=3) == [\"2 x done\", \"3 x done\", \"4 x done\"]\n    assert r.tick(60) == [\"5 x done\", \"6 x done\", \"1 x done\"] and done == [0, 1, 2, 3, 4, \"late\"]")

## Warm-up

A `deque` (Week 43) is a fine in-process queue: add at one end, take from the other, both in constant time. A **message queue** in a real system is the same idea, run as a separate service so that the program adding work and the programs doing it can be on different machines, run at different speeds, and fail independently.

## Why queues

A **producer** puts **messages** (small descriptions of work: "send the welcome email to user 7") on a **queue**; **consumers** (workers) take them off and do the work. The queue sits between them and buys four things:

- **Fast requests**: the request only has to enqueue, which takes a millisecond; the slow work happens afterwards.
- **Decoupling**: if the email provider is down, the tickets still get saved; the messages wait.
- **Smoothing bursts**: a spike of 10,000 sign-ups becomes a queue that workers drain at a steady rate (Monday's queueing, used on purpose), instead of 10,000 slow requests at once.
- **Retries**: a message whose work failed goes back on the queue to try again later.

What should go on a queue? Work that the user doesn't need to see finished before getting a response: emails and notifications, webhooks (Week 80), image processing, reports, search indexing. What shouldn't? Anything the response depends on: you can't queue "check the password".

**Backpressure.** A queue that can grow without limit just moves the problem: producers keep adding, memory fills, and the backlog takes hours to drain. Real queues are **bounded**: when full, the producer is told **no** (the queue refuses, or the API answers `429 Too Many Requests` or `503`), so the slowdown is visible and handled where it starts.

## Delivery, acknowledgement and retries

A worker can crash halfway through a message. What happens to it decides the **delivery guarantee**:

- **At most once**: the message is removed as soon as it's handed out. A crash loses it. Fine for metrics, not for emails people are waiting for.
- **At least once**: the message stays in the queue, **invisible**, while a worker has it. The worker **acknowledges** (acks) it only after finishing the work, and then it's deleted. If no ack arrives within the **visibility timeout**, the message reappears for another worker. Nothing is lost, but a slow (not crashed) worker can cause the same message to be processed **twice**.
- **Exactly once** is what everyone wants, and over a network it can't be guaranteed by delivery alone; systems get its **effect** with at-least-once delivery plus **idempotent** consumers (next section).

Messages that fail again and again (a "poison" message, such as an email to an address that doesn't exist) must not cycle forever, blocking the queue. After a maximum number of receives, the queue moves them to a **dead-letter queue** (DLQ), where a person can look at them. Retries should wait longer each time (**exponential backoff**, Week 78), so a struggling downstream service gets room to recover.

## Idempotent consumers

With at-least-once delivery, **duplicates will happen**: a worker finishes the work, then crashes before its ack; a network hiccup makes the producer send twice. So consumers must be **idempotent** (Week 77's idempotency keys, from the other side): processing the same message twice has the same effect as once. Two ways:

- **Naturally idempotent operations**: "set the ticket's status to resolved" can be repeated harmlessly; "add 1 to the counter" can't.
- **Deduplication**: every message carries a unique **id**; the consumer records the ids it has processed (in the same transaction as the work, ideally), and skips any it has seen. Keep ids for longer than a duplicate could plausibly arrive (a day, say), then forget them.

## Fan-out and partitions

**Publish and subscribe** (pub/sub): sometimes one event interests several consumers. "Ticket resolved" should send an email, notify the chat system and update the statistics. With pub/sub, the producer **publishes** to a **topic**, and each **subscription** gets its **own copy** in its own queue. Subscribers are added without changing the producer, and a slow subscriber doesn't hold up the others.

**Ordering.** A single queue with many workers doesn't keep order: message 2 can finish before message 1. Often that's fine; sometimes it isn't ("ticket created" must be handled before "ticket closed"). **Partitioned logs**, the design behind Kafka, solve it at scale: a topic is split into **partitions**, each message goes to the partition chosen by a **stable hash of its key** (the ticket id), so all of one ticket's events are in one partition, in order; and each partition is read by one consumer at a time. Consumers track their position, the **offset**, per partition, so they can resume after a restart or re-read history. The hash must be **stable** across processes and restarts: Python's built-in `hash()` of a string changes every time Python starts, so use something like `zlib.crc32`.

### On your own computer

Common message systems: **RabbitMQ** (classic queues and routing), **Amazon SQS** (visibility timeouts and dead-letter queues exactly as above), **Redis** (lists and streams for simple cases) and **Apache Kafka** (partitioned logs). In Python, **Celery** and **RQ** run background jobs on top of them: you decorate a function as a task and call `send_welcome.delay(user_id)`, and a separate worker process runs it.

## Practice

## Quiz

## Daily challenge

## Before you go

Think of three things the helpdesk does during a request that could move to a queue, and one that never could. Tomorrow: **databases at scale**: replicas, partitioning, sharding, and the trade-offs that come with spreading data over many machines.